In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"

import zipfile
import io
import requests

response = requests.get(url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("student-mat.csv") as file:
        df = pd.read_csv(file, sep=";")
print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
df.head()

In [ ]:
print("Number of rows and columns:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nDataset information:")
df.info()
print("\nStatistical summary:")
display(df.describe())
print("\nFirst 10 student records:")
display(df.head(10))

In [ ]:
print("Missing values:")
print(df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Dataset shape after cleaning:", df.shape)

In [ ]:
print("Average first-period grade:", round(df["G1"].mean(), 2))
print("Average second-period grade:", round(df["G2"].mean(), 2))
print("Average final grade:", round(df["G3"].mean(), 2))
print("\nHighest final grade:", df["G3"].max())
print("Lowest final grade:", df["G3"].min())
print("Students scoring 10 or above:", (df["G3"] >= 10).sum())
print("Students scoring below 10:", (df["G3"] < 10).sum())

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df["G3"], bins=20, kde=True)
plt.title("Distribution of Student Final Grades")
plt.xlabel("Final Grade")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
average_grades = df[["G1", "G2", "G3"]].mean()

plt.figure(figsize=(8, 5))
average_grades.plot(kind="bar")
plt.title("Average Student Grades by Period")
plt.xlabel("Exam Period")
plt.ylabel("Average Grade out of 20")
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=df, x="studytime", y="G3", errorbar=None)
plt.title("Study Time vs Final Grade")
plt.xlabel("Weekly Study Time Category")
plt.ylabel("Average Final Grade")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="absences", y="G3")
plt.title("School Absences vs Final Grade")
plt.xlabel("Number of Absences")
plt.ylabel("Final Grade")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Between Student Performance Factors")
plt.show()

In [ ]:
gender_performance = df.groupby("sex")["G3"].mean()

print(gender_performance)

plt.figure(figsize=(6, 4))
gender_performance.plot(kind="bar")
plt.title("Average Final Grade by Gender")
plt.xlabel("Gender (F = Female, M = Male)")
plt.ylabel("Average Final Grade")
plt.xticks(rotation=0)
plt.show()

In [ ]:
df["Performance_Status"] = np.where(
    df["G3"] < 10,
    "Needs Support",
    "Pass"
)

print(df["Performance_Status"].value_counts())

sns.countplot(
    data=df,
    x="Performance_Status",
    hue="Performance_Status",
    legend=False
)

plt.title("Student Performance Status")
plt.xlabel("Performance Status")
plt.ylabel("Number of Students")
plt.show()

In [ ]:
# Target variable
y = df["G3"]

# Exclude final grade and earlier exam grades from the inputs
X = df.drop(
    columns=["G1", "G2", "G3", "Performance_Status"]
)

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

# Preprocessing pipelines
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

# Build the complete model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# Train the model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("Model training completed!")

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", round(mae, 2))
print("Root Mean Squared Error (RMSE):", round(rmse, 2))
print("R-squared Score:", round(r2, 3))

In [ ]:
results = pd.DataFrame({
    "Actual Grade": y_test.values,
    "Predicted Grade": y_pred.round(2)
})

display(results.head(15))

plt.figure(figsize=(7, 6))
sns.scatterplot(
    x=y_test,
    y=y_pred
)

plt.plot([0, 20], [0, 20], linestyle="--")
plt.xlim(0, 20)
plt.ylim(0, 20)
plt.xlabel("Actual Final Grade")
plt.ylabel("Predicted Final Grade")
plt.title("Actual vs Predicted Student Grades")
plt.show()

In [ ]:
new_student = X.iloc[[0]].copy()

# Modify a few example characteristics
new_student.loc[:, "studytime"] = 4
new_student.loc[:, "absences"] = 2
new_student.loc[:, "failures"] = 0

predicted_grade = model.predict(new_student)[0]

print("Predicted final grade:", round(predicted_grade, 2), "/ 20")

In [ ]:
print("=" * 45)
print("STUDENT PERFORMANCE ANALYSIS REPORT")
print("=" * 45)

print("Total students analyzed:", len(df))
print("Average final grade:", round(df["G3"].mean(), 2))
print("Highest final grade:", df["G3"].max())
print("Lowest final grade:", df["G3"].min())

print(
    "Students scoring 10 or above:",
    int((df["G3"] >= 10).sum())
)

print(
    "Students scoring below 10:",
    int((df["G3"] < 10).sum())
)

print("Model MAE:", round(mae, 2))
print("Model RMSE:", round(rmse, 2))
print("Model R²:", round(r2, 3))

print("=" * 45)
print("Analysis completed successfully!")